## Day 1 — What is Apache Spark?

### The Big Idea

Spark is a **distributed compute engine**. You write Python (or SQL), and Spark runs it across a cluster of machines in parallel.

Think of it as: pandas, but the data is split across 10 (or 1000) machines and processed simultaneously.

---

### Core Concepts

#### 1. Driver vs Executors

```
Your Notebook (Driver)
        │
        ▼
   Spark Context
        │
   ┌────┴────┐
   ▼         ▼
Executor1  Executor2  ...  ExecutorN
(Worker)   (Worker)        (Worker)
```

- **Driver:** your code, your notebook — orchestrates the job, holds the plan
- **Executors:** worker nodes that actually process data — each handles a partition
- **SparkContext / SparkSession:** the entry point that connects driver to the cluster

In Databricks, `spark` is already available — no setup needed.

---

#### 2. Lazy Evaluation

Spark does **nothing** until you force it to.

- `select()`, `filter()`, `withColumn()` → **transformations** — just build a plan, no data moves
- `show()`, `count()`, `write()` → **actions** — trigger actual execution

Why? So Spark can optimize the full plan before running anything.

---

#### 3. RDD → DataFrame → Dataset

| Layer | What it is | Used today? |
|---|---|---|
| RDD | Raw distributed collection, low-level | Rarely |
| DataFrame | Distributed table with schema (like pandas) | Yes — this is your main tool |
| Dataset | Typed DataFrame (Scala/Java only) | Not in Python |

You will almost always work with **DataFrames** and **Spark SQL**.

In [ ]:
# Check Spark version — confirm the cluster is attached
print(spark.version)

In [ ]:
# Create a small DataFrame from scratch
data = [
    ("Alice", 34, "Engineering"),
    ("Bob",   28, "Marketing"),
    ("Carol", 41, "Engineering"),
    ("Dave",  25, "Sales"),
]
columns = ["name", "age", "department"]

df = spark.createDataFrame(data, columns)
df.show()

In [ ]:
# Inspect schema — Spark inferred types automatically
df.printSchema()

In [ ]:
# Lazy evaluation demo
# This builds a plan — nothing runs yet
filtered = df.filter(df.age > 30).select("name", "department")

# THIS triggers execution
filtered.show()

In [ ]:
# See the execution plan Spark built
filtered.explain()

---

## Day 2 — Lazy Evaluation Deep Dive

### Transformations vs Actions

| Transformations (lazy) | Actions (trigger execution) |
|---|---|
| `select()` | `show()` |
| `filter()` / `where()` | `count()` |
| `withColumn()` | `collect()` |
| `groupBy()` | `write()` |
| `join()` | `toPandas()` |
| `orderBy()` | `first()` / `take()` |

Chaining transformations is free — Spark just builds a DAG (Directed Acyclic Graph) of operations. The moment you call an action, Spark optimizes and executes the whole chain.

### Why this matters:
- You can build complex multi-step pipelines without wasting compute on intermediate results
- Spark's **Catalyst optimizer** rewrites your plan to be more efficient before running it
- `explain()` lets you see what Spark actually plans to do

In [ ]:
from pyspark.sql import functions as F

# Build a multi-step plan — nothing executes yet
result = (
    df
    .filter(F.col("age") > 25)
    .withColumn("age_group", F.when(F.col("age") >= 35, "Senior").otherwise("Junior"))
    .groupBy("department", "age_group")
    .agg(F.count("*").alias("headcount"))
    .orderBy("department")
)

# Action — now Spark runs the whole thing
result.show()

In [ ]:
# count() is an action — triggers execution and returns a Python int
n = df.count()
print(f"Total rows: {n}")

---

## ✅ Days 1–2 Checklist

- [ ] `spark.version` printed successfully
- [ ] Created a DataFrame with `spark.createDataFrame()`
- [ ] Used `show()`, `printSchema()`, `explain()`
- [ ] Understood: transformations are lazy, actions trigger execution
- [ ] Chained multiple transformations and ran as one job

**Next:** Day 3 — Spark Runtime Architecture (Jobs, Stages, Tasks)